## RANDOM FOREST

Trying non-linear models

In [1]:
SEED = 42

In [2]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import PCA

import pandas as pd
import numpy as np
from utils import MLUtils 

import pickle

In [3]:
de_train_path = "/Users/jennyng/Desktop/Projects/kaggle/single-cell-perturbations/data/de_train.parquet"
de_train = pd.read_parquet(de_train_path)
de_train.head()

,cell_type,sm_name,sm_lincs_id,SMILES,control,A1BG,A1BG-AS1,A2M,A2M-AS1,A2MP1,...,ZUP1,ZW10,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11B,ZYX,ZZEF1
0,NK cells,Clotrimazole,LSM-5341,Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1,False,0.104720,-0.077524,-1.625596,-0.144545,0.143555,...,-0.227781,-0.010752,-0.023881,0.674536,-0.453068,0.005164,-0.094959,0.034127,0.221377,0.368755
1,T cells CD4+,Clotrimazole,LSM-5341,Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1,False,0.915953,-0.884380,0.371834,-0.081677,-0.498266,...,-0.494985,-0.303419,0.304955,-0.333905,-0.315516,-0.369626,-0.095079,0.704780,1.096702,-0.869887
2,T cells CD8+,Clotrimazole,LSM-5341,Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1,False,-0.387721,-0.305378,0.567777,0.303895,-0.022653,...,-0.119422,-0.033608,-0.153123,0.183597,-0.555678,-1.494789,-0.213550,0.415768,0.078439,-0.259365
3,T regulatory cells,Clotrimazole,LSM-5341,Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1,False,0.232893,0.129029,0.336897,0.486946,0.767661,...,0.451679,0.704643,0.015468,-0.103868,0.865027,0.189114,0.224700,-0.048233,0.216139,-0.085024
4,NK cells,Mometasone Furoate,LSM-3349,C[C@@H]1C[C@H]2[C@@H]3CCC4=CC(=O)C=C[C@]4(C)[C...,False,4.290652,-0.063864,-0.017443,-0.541154,0.570982,...,0.758474,0.510762,0.607401,-0.123059,0.214366,0.487838,-0.819775,0.112365,-0.122193,0.676629


In [4]:
path = "/Users/jennyng/Desktop/Projects/kaggle/single-cell-perturbations/processed_data/train_test_splits.pkl"
with open(path, "rb") as f:
    folds = pickle.load(f)

In [5]:
# load one hot cell encodings
ohe_cell = pd.read_csv("/Users/jennyng/Desktop/Projects/kaggle/single-cell-perturbations/processed_data/onehot_cell_type.csv", index_col="cell_type")
ohe_cell

# convert cell encodings to dict
ohe_cell_map = dict(zip(ohe_cell.index, ohe_cell.to_numpy()))
ohe_cell_map

{'B cells': array([1, 0, 0, 0, 0, 0]),
 'Myeloid cells': array([0, 1, 0, 0, 0, 0]),
 'NK cells': array([0, 0, 1, 0, 0, 0]),
 'T cells CD4+': array([0, 0, 0, 1, 0, 0]),
 'T cells CD8+': array([0, 0, 0, 0, 1, 0]),
 'T regulatory cells': array([0, 0, 0, 0, 0, 1])}

In [6]:
# load one hot drug encodings
ohe_sm = pd.read_csv("/Users/jennyng/Desktop/Projects/kaggle/single-cell-perturbations/processed_data/onehot_sm_name.csv", index_col="sm_name")
ohe_sm

# convert to dict
ohe_sm_map = dict(zip(ohe_sm.index, ohe_sm.to_numpy()))
ohe_sm_map

{'5-(9-Isopropyl-8-methyl-2-morpholino-9H-purin-6-yl)pyrimidin-2-amine': array([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 'ABT-199 (GDC-0199)': array([0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [ ]:
X_tr, y_tr = MLUtils.featurize_cell_sm(de_train, ohe_cell_map, ohe_sm_map)
rf_model = RandomForestRegressor(random_state=SEED)
pca_y = PCA(n_components=50)    # fitting on raw GEX takes a long time, so we reduce dimensionality first

MLUtils.train(X_tr, y_tr, rf_model, folds, pca_y)

{'folds': [1.1403280852898228,
  1.1433384227456054,
  1.5068168612127022,
  1.3202554019230792,
  1.385570805525713,
  1.113714563631556,
  0.9319725989408499,
  1.0518198412490631,
  1.2073355510894408,
  1.4812168937545973],
 'mean': 1.2282369025362432}

In [11]:
mrrmse_list = []
alpha_list = []

for i, (train_idx, test_idx) in enumerate(folds):
    X_train = X_tr.iloc[train_idx]
    y_train = y_tr.iloc[train_idx]
    X_test = X_tr.iloc[test_idx]
    y_test = y_tr.iloc[test_idx]

    y_hat_baseline = y_train.median()

    y_train_res = y_train - y_hat_baseline

    # fit model trained on residual
    pca_y_res = PCA(n_components=50)
    z_train_res = pca_y_res.fit_transform(y_train_res)

    res_model = RandomForestRegressor()
    res_model.fit(X_train, z_train_res)

    z_hat_res = res_model.predict(X_test)
    y_hat_res = pca_y_res.inverse_transform(z_hat_res)

    y_hat = y_hat_res + y_hat_baseline.to_numpy()

    err = MLUtils.mrrmse(y_test, y_hat)
    print(f"fold {i}, mrrmse = {err}")
    mrrmse_list.append(err)

avg_mrrmse = sum(mrrmse_list)/len(mrrmse_list)
print(f"avg MRRMSE = {avg_mrrmse}")

fold 0, mrrmse = 1.1436010760769268
fold 1, mrrmse = 1.1376778550525568
fold 2, mrrmse = 1.5251974831347832
fold 3, mrrmse = 1.315642703677373
fold 4, mrrmse = 1.3853513424230617
fold 5, mrrmse = 1.1116925088285905
fold 6, mrrmse = 0.9278734947760092
fold 7, mrrmse = 1.0665544604578345
fold 8, mrrmse = 1.2555871516001202
fold 9, mrrmse = 1.4983233051339773
avg MRRMSE = 1.2367501381161234
